# Day 2: Data Ingestion and Validation

## Project
**IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning**

### Day 2 objective

Build and validate the first ingestion layer using the **extracted source files**.

Day 2 focuses on:

1. Inspecting the actual source schemas again before transformation.
2. Investigating the 51 exact duplicate ball-by-ball rows found on Day 1.
3. Reconciling the 74 extracted 2026 match directories with the 70-row `fixtures.csv`.
4. Deriving complete match metadata from authoritative match-level data.
5. Inspecting player identity and Cricsheet registry IDs.
6. Defining validation rules.
7. Creating normalized processed match metadata.
8. Preparing the reusable ingestion pipeline structure.
9. Verifying that raw data is never modified.

### Day 2 rule

Do not silently delete records or invent metadata.

Every transformation must be traceable to an observed source field or a documented rule.


## Expected source structure

```text
data/
├── raw/
│   ├── ipl_male_json/
│   │   ├── README.txt
│   │   └── *.json
│   │
│   └── 2026/
│       ├── fixtures.csv
│       └── matches/
│           ├── match_01_.../
│           └── ...
│
└── processed/
```

The raw directories are read-only for this notebook.


In [1]:
from pathlib import Path
import json
import re
import hashlib
import pandas as pd
import numpy as np
from IPython.display import display

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
CRICSHEET_DIR = RAW_DIR / "ipl_male_json"
DATA_2026_DIR = RAW_DIR / "2026"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Cricsheet:", CRICSHEET_DIR)
print("2026 data:", DATA_2026_DIR)
print("Processed output:", PROCESSED_DIR)

assert CRICSHEET_DIR.exists()
assert DATA_2026_DIR.exists()


Project root: d:\Git\Personal\ipl-player-performance
Cricsheet: d:\Git\Personal\ipl-player-performance\data\raw\ipl_male_json
2026 data: d:\Git\Personal\ipl-player-performance\data\raw\2026
Processed output: d:\Git\Personal\ipl-player-performance\data\processed


## 1. Source inventory and schema inspection

In [2]:
json_files = sorted(CRICSHEET_DIR.glob("*.json"))
match_dirs = sorted(
    p for p in (DATA_2026_DIR / "matches").iterdir()
    if p.is_dir()
)

fixtures_path = DATA_2026_DIR / "fixtures.csv"
fixtures = pd.read_csv(fixtures_path)

print("Cricsheet JSON matches:", len(json_files))
print("2026 match directories:", len(match_dirs))
print("2026 fixture rows:", len(fixtures))

print("\nFixture columns:")
print(fixtures.columns.tolist())

# Inspect one complete 2026 match directory.
sample_match_dir = match_dirs[0]
print("\nSample match directory:", sample_match_dir.name)

for path in sorted(sample_match_dir.iterdir()):
    if path.is_file():
        print(f"  {path.name:30} {path.stat().st_size:,} bytes")


Cricsheet JSON matches: 1243
2026 match directories: 74
2026 fixture rows: 70

Fixture columns:
['match_number', 'date', 'venue', 'team_1', 'team_2']

Sample match directory: match_01_Sunrisers_Hyderabad_vs_Royal_Challengers_Bengaluru
  ball_by_ball.csv               25,271 bytes
  batting_scorecard.csv          1,360 bytes
  bowling_scorecard.csv          806 bytes
  fall_of_wickets.csv            663 bytes
  partnerships.csv               1,119 bytes
  phase_summary.csv              383 bytes
  reviews.csv                    171 bytes
  substitutions.csv              180 bytes


## 2. Duplicate ball-by-ball investigation

Day 1 found **51 exact duplicate rows across 36 match files**.

The objective here is to determine whether:

- the same complete delivery appears twice in the source, or
- the apparent duplicate is caused by a missing match/file key in the comparison.

No rows are deleted in this step.


In [3]:
ball_paths = sorted(DATA_2026_DIR.rglob("ball_by_ball.csv"))

duplicate_details = []

for path in ball_paths:
    df = pd.read_csv(path)

    dup_mask = df.duplicated(keep=False)

    if dup_mask.any():
        duplicate_rows = df.loc[dup_mask].copy()

        duplicate_details.append({
            "source_file": str(path.relative_to(DATA_2026_DIR)),
            "total_rows": len(df),
            "duplicate_rows_including_repeats": len(duplicate_rows),
            "duplicate_groups": int(
                df.duplicated(keep=False).sum()
            ),
        })

duplicate_summary = pd.DataFrame(duplicate_details)

print("Files containing exact duplicate rows:", len(duplicate_summary))

if not duplicate_summary.empty:
    display(duplicate_summary)

print(
    "\nTotal duplicate rows including repeated copies:",
    duplicate_summary["duplicate_rows_including_repeats"].sum()
    if not duplicate_summary.empty else 0
)


Files containing exact duplicate rows: 36


,source_file,total_rows,duplicate_rows_including_repeats,duplicate_groups
0,matches\match_05_Lucknow_Super_Giants_vs_Delhi...,229,2,2
1,matches\match_06_Sunrisers_Hyderabad_vs_Kolkat...,231,2,2
2,matches\match_07_Chennai_Super_Kings_vs_Punjab...,251,2,2
3,matches\match_14_Gujarat_Titans_vs_Delhi_Capit...,254,2,2
4,matches\match_17_Sunrisers_Hyderabad_vs_Punjab...,250,5,5
5,matches\match_20_Royal_Challengers_Bengaluru_v...,262,2,2
6,matches\match_21_Sunrisers_Hyderabad_vs_Rajast...,251,2,2
7,matches\match_22_Chennai_Super_Kings_vs_Kolkat...,255,2,2
8,matches\match_27_Sunrisers_Hyderabad_vs_Chenna...,257,2,2
9,matches\match_28_Rajasthan_Royals_vs_Kolkata_K...,256,4,4



Total duplicate rows including repeated copies: 99


In [4]:
# Inspect representative duplicate groups.

for path in ball_paths:
    df = pd.read_csv(path)
    dup_mask = df.duplicated(keep=False)

    if dup_mask.any():
        duplicate_rows = df.loc[dup_mask].copy()

        print("\nSOURCE:", path.relative_to(DATA_2026_DIR))
        display(duplicate_rows.head(10))

        # Inspect only the first affected file.
        break



SOURCE: matches\match_05_Lucknow_Super_Giants_vs_Delhi_Capitals\ball_by_ball.csv


,innings,team,over,ball,batter,bowler,non_striker,batter_runs,extra_runs,total_runs,...,wides,noballs,byes,legbyes,penalty,is_wicket,wicket_kind,player_out,fielder,phase
115,2,Delhi Capitals,1,2.wides,N Rana,Mohammed Shami,P Nissanka,0,1,1,...,1,0,0,0,0,False,NaN,NaN,NaN,powerplay
116,2,Delhi Capitals,1,2.wides,N Rana,Mohammed Shami,P Nissanka,0,1,1,...,1,0,0,0,0,False,NaN,NaN,NaN,powerplay


### Duplicate decision

Do not drop duplicates yet.

The Day 2 ingestion rule will preserve source rows until a stable match key and delivery key are defined.

The final cleaning rule must be based on actual source semantics, not simply `drop_duplicates()`.


## 3. Reconcile 74 match directories vs 70 fixtures

`fixtures.csv` does not contain metadata rows for every extracted 2026 match directory.

The authoritative match-level metadata will therefore be derived from the Cricsheet JSON source for 2026 and compared with the processed archive.


In [5]:
# Extract 2026 match metadata from Cricsheet.

cricsheet_2026_rows = []

for path in json_files:
    with path.open("r", encoding="utf-8") as f:
        match = json.load(f)

    info = match.get("info", {})
    season = str(info.get("season", "")).strip()

    if season != "2026":
        continue

    dates = info.get("dates", [])
    date_value = str(dates[0]) if dates else None

    teams = info.get("teams", [])

    cricsheet_2026_rows.append({
        "cricsheet_file": path.name,
        "date": date_value,
        "season": season,
        "venue": info.get("venue"),
        "city": info.get("city"),
        "team_1": teams[0] if len(teams) > 0 else None,
        "team_2": teams[1] if len(teams) > 1 else None,
        "match_type": info.get("type"),
        "match_id": path.stem,
    })

cricsheet_2026 = pd.DataFrame(cricsheet_2026_rows)

cricsheet_2026["date"] = pd.to_datetime(
    cricsheet_2026["date"],
    errors="coerce"
)

print("Cricsheet 2026 matches:", len(cricsheet_2026))
display(cricsheet_2026.head())


Cricsheet 2026 matches: 74


,cricsheet_file,date,season,venue,city,team_1,team_2,match_type,match_id
0,1527674.json,2026-03-28,2026,"M Chinnaswamy Stadium, Bengaluru",Bengaluru,Sunrisers Hyderabad,Royal Challengers Bengaluru,None,1527674
1,1527675.json,2026-03-29,2026,"Wankhede Stadium, Mumbai",Mumbai,Kolkata Knight Riders,Mumbai Indians,None,1527675
2,1527676.json,2026-03-30,2026,"Barsapara Cricket Stadium, Guwahati",Guwahati,Chennai Super Kings,Rajasthan Royals,None,1527676
3,1527677.json,2026-03-31,2026,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Gujarat Titans,Punjab Kings,None,1527677
4,1527678.json,2026-04-01,2026,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Lucknow Super Giants,Delhi Capitals,None,1527678


In [6]:
# Inspect the four matches beyond fixture row 70.

fixture_dates = pd.to_datetime(
    fixtures["date"],
    errors="coerce"
)

fixture_pairs = (
    fixtures.assign(date=fixture_dates)
    [["date", "team_1", "team_2"]]
)

print("Cricsheet 2026 date range:")
print(
    cricsheet_2026["date"].min(),
    "to",
    cricsheet_2026["date"].max()
)

print("\nFixture date range:")
print(
    fixture_dates.min(),
    "to",
    fixture_dates.max()
)

# Show the latest Cricsheet matches.
display(
    cricsheet_2026
    .sort_values("date")
    .tail(10)
)


Cricsheet 2026 date range:
2026-03-28 00:00:00 to 2026-05-31 00:00:00

Fixture date range:
2026-03-28 00:00:00 to 2026-05-24 00:00:00


,cricsheet_file,date,season,venue,city,team_1,team_2,match_type,match_id
64,1529308.json,2026-05-20,2026,"Eden Gardens, Kolkata",Kolkata,Mumbai Indians,Kolkata Knight Riders,None,1529308
65,1529309.json,2026-05-21,2026,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,Gujarat Titans,Chennai Super Kings,None,1529309
66,1529310.json,2026-05-22,2026,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Hyderabad,Sunrisers Hyderabad,Royal Challengers Bengaluru,None,1529310
67,1529311.json,2026-05-23,2026,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Lucknow Super Giants,Punjab Kings,None,1529311
68,1529312.json,2026-05-24,2026,"Wankhede Stadium, Mumbai",Mumbai,Rajasthan Royals,Mumbai Indians,None,1529312
69,1529313.json,2026-05-24,2026,"Eden Gardens, Kolkata",Kolkata,Delhi Capitals,Kolkata Knight Riders,None,1529313
70,1535462.json,2026-05-26,2026,"Himachal Pradesh Cricket Association Stadium, ...",Dharamsala,Royal Challengers Bengaluru,Gujarat Titans,None,1535462
71,1535463.json,2026-05-27,2026,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Rajasthan Royals,Sunrisers Hyderabad,None,1535463
72,1535464.json,2026-05-29,2026,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Rajasthan Royals,Gujarat Titans,None,1535464
73,1535465.json,2026-05-31,2026,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,Gujarat Titans,Royal Challengers Bengaluru,None,1535465


## 4. Team-name normalization for reconciliation

Team names can differ in formatting between sources.

Examples that require inspection include:

- `Rajasthan Royals`
- `RajasthanRoyals`

The normalization function below is only for comparison. It does not overwrite the original source values.


In [7]:
def normalize_team_name(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    # Comparison-only normalization.
    value = re.sub(r"[^a-z0-9]", "", value)

    return value


fixtures_compare = fixtures.copy()
fixtures_compare["date"] = pd.to_datetime(
    fixtures_compare["date"],
    errors="coerce"
)

for df in [fixtures_compare, cricsheet_2026]:
    df["team_1_norm"] = df["team_1"].map(normalize_team_name)
    df["team_2_norm"] = df["team_2"].map(normalize_team_name)

fixtures_compare["team_pair_norm"] = (
    fixtures_compare["team_1_norm"]
    + "__"
    + fixtures_compare["team_2_norm"]
).map(str)

cricsheet_2026["team_pair_norm"] = (
    cricsheet_2026["team_1_norm"]
    + "__"
    + cricsheet_2026["team_2_norm"]
).map(str)

print("Normalization created for comparison only.")


Normalization created for comparison only.


In [8]:
# Reconcile using date + unordered normalized team pair.

def unordered_pair(a, b):
    return "||".join(sorted([str(a), str(b)]))

fixtures_compare["match_key"] = fixtures_compare.apply(
    lambda r: (
        f"{r['date'].date()}__"
        f"{unordered_pair(r['team_1_norm'], r['team_2_norm'])}"
    ),
    axis=1
)

cricsheet_2026["match_key"] = cricsheet_2026.apply(
    lambda r: (
        f"{r['date'].date()}__"
        f"{unordered_pair(r['team_1_norm'], r['team_2_norm'])}"
    ),
    axis=1
)

fixture_keys = set(fixtures_compare["match_key"])
cricsheet_keys = set(cricsheet_2026["match_key"])

missing_from_fixtures = cricsheet_2026[
    ~cricsheet_2026["match_key"].isin(fixture_keys)
].copy()

missing_from_cricsheet = fixtures_compare[
    ~fixtures_compare["match_key"].isin(cricsheet_keys)
].copy()

print("Cricsheet 2026 matches:", len(cricsheet_2026))
print("Fixture rows:", len(fixtures_compare))
print("Cricsheet matches absent from fixtures:", len(missing_from_fixtures))
print("Fixture rows absent from Cricsheet:", len(missing_from_cricsheet))

print("\nMatches found in Cricsheet but absent from fixtures.csv:")
display(
    missing_from_fixtures[
        ["date", "venue", "city", "team_1", "team_2", "match_id"]
    ]
)

print("\nFixture rows not matched to Cricsheet:")
display(
    missing_from_cricsheet[
        ["date", "venue", "team_1", "team_2"]
    ]
)


Cricsheet 2026 matches: 74
Fixture rows: 70
Cricsheet matches absent from fixtures: 4
Fixture rows absent from Cricsheet: 0

Matches found in Cricsheet but absent from fixtures.csv:


,date,venue,city,team_1,team_2,match_id
70,2026-05-26,"Himachal Pradesh Cricket Association Stadium, ...",Dharamsala,Royal Challengers Bengaluru,Gujarat Titans,1535462
71,2026-05-27,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Rajasthan Royals,Sunrisers Hyderabad,1535463
72,2026-05-29,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Rajasthan Royals,Gujarat Titans,1535464
73,2026-05-31,"Narendra Modi Stadium, Ahmedabad",Ahmedabad,Gujarat Titans,Royal Challengers Bengaluru,1535465



Fixture rows not matched to Cricsheet:


,date,venue,team_1,team_2


## 5. Player identity inspection

Cricsheet contains player names and registry information.

Before creating a player dimension, inspect:

- player names
- registry person IDs
- repeated names
- naming differences between the Cricsheet and 2026 processed sources

No player ID is invented in Day 2.


In [9]:
# Build a Cricsheet player registry audit.

player_registry_rows = []

for path in json_files:
    with path.open("r", encoding="utf-8") as f:
        match = json.load(f)

    info = match.get("info", {})
    registry = info.get("registry", {})
    people = registry.get("people", {})
    players = info.get("players", {})

    for team, team_players in players.items():
        for player in team_players:
            player_registry_rows.append({
                "player_name": player,
                "team": team,
                "registry_id": people.get(player),
                "source_match": path.stem,
            })

player_registry = pd.DataFrame(player_registry_rows)

print("Player-team observations:", len(player_registry))
print("Unique player names:", player_registry["player_name"].nunique())
print(
    "Unique registry IDs:",
    player_registry["registry_id"].dropna().nunique()
)

display(
    player_registry.head(20)
)


Player-team observations: 27909
Unique player names: 811
Unique registry IDs: 809


,player_name,team,registry_id,source_match
0,CH Gayle,Royal Challengers Bangalore,db584dad,1082591
1,Mandeep Singh,Royal Challengers Bangalore,c3a96caf,1082591
2,TM Head,Royal Challengers Bangalore,12b610c2,1082591
3,KM Jadhav,Royal Challengers Bangalore,99d63244,1082591
4,SR Watson,Royal Challengers Bangalore,4329fbb5,1082591
5,Sachin Baby,Royal Challengers Bangalore,dc9dd038,1082591
6,STR Binny,Royal Challengers Bangalore,bd17b45f,1082591
7,S Aravind,Royal Challengers Bangalore,957532de,1082591
8,TS Mills,Royal Challengers Bangalore,245c97cb,1082591
9,YS Chahal,Royal Challengers Bangalore,57ee1fde,1082591


In [10]:
# Check names that map to more than one registry ID.

name_id_counts = (
    player_registry
    .dropna(subset=["registry_id"])
    .groupby("player_name")["registry_id"]
    .nunique()
    .sort_values(ascending=False)
)

ambiguous_names = name_id_counts[name_id_counts > 1]

print(
    "Player names mapped to multiple registry IDs:",
    len(ambiguous_names)
)

if len(ambiguous_names):
    display(ambiguous_names.to_frame("Registry ID Count").head(20))
else:
    print("No ambiguous player-name to registry-ID mappings observed.")


Player names mapped to multiple registry IDs: 1


,Registry ID Count
player_name,
Harmeet Singh,2


## 6. Source validation rules

Day 2 establishes the following validation rules.

### Match-level

- Every processed match must have a stable source match identifier.
- Match date must be parseable.
- At least two teams must be present.
- Match metadata must be traceable to the source.
- The 2026 fixture file is not treated as the complete authority.

### Ball-level

- `batter`, `bowler`, and `non_striker` must be inspectable.
- Delivery ordering must be preserved.
- Duplicate records must be investigated before removal.
- Numeric run fields must be validated as numeric.

### Player-level

- Player names must be standardized without destroying source names.
- Cricsheet registry IDs should be retained where available.
- No fabricated IDs are allowed.

### Reproducibility

- Raw data is read-only.
- Processed outputs go under `data/processed/`.
- Transformations must be deterministic.


## 7. Create normalized 2026 match metadata

The normalized metadata below is a Day 2 processed artifact.

The original source values are retained, while normalized comparison fields are separate.


In [11]:
normalized_match_metadata = cricsheet_2026[
    [
        "match_id",
        "cricsheet_file",
        "date",
        "season",
        "venue",
        "city",
        "team_1",
        "team_2",
        "match_type",
    ]
].copy()

normalized_match_metadata["team_1_norm"] = (
    normalized_match_metadata["team_1"]
    .map(normalize_team_name)
)

normalized_match_metadata["team_2_norm"] = (
    normalized_match_metadata["team_2"]
    .map(normalize_team_name)
)

normalized_match_metadata["match_key"] = normalized_match_metadata.apply(
    lambda r: (
        f"{r['date'].date()}__"
        f"{unordered_pair(r['team_1_norm'], r['team_2_norm'])}"
    ),
    axis=1
)

output_path = PROCESSED_DIR / "match_metadata_2026.csv"

normalized_match_metadata.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(normalized_match_metadata))

display(normalized_match_metadata.head())


Saved: d:\Git\Personal\ipl-player-performance\data\processed\match_metadata_2026.csv
Rows: 74


,match_id,cricsheet_file,date,season,venue,city,team_1,team_2,match_type,team_1_norm,team_2_norm,match_key
0,1527674,1527674.json,2026-03-28,2026,"M Chinnaswamy Stadium, Bengaluru",Bengaluru,Sunrisers Hyderabad,Royal Challengers Bengaluru,None,sunrisershyderabad,royalchallengersbengaluru,2026-03-28__royalchallengersbengaluru||sunrise...
1,1527675,1527675.json,2026-03-29,2026,"Wankhede Stadium, Mumbai",Mumbai,Kolkata Knight Riders,Mumbai Indians,None,kolkataknightriders,mumbaiindians,2026-03-29__kolkataknightriders||mumbaiindians
2,1527676,1527676.json,2026-03-30,2026,"Barsapara Cricket Stadium, Guwahati",Guwahati,Chennai Super Kings,Rajasthan Royals,None,chennaisuperkings,rajasthanroyals,2026-03-30__chennaisuperkings||rajasthanroyals
3,1527677,1527677.json,2026-03-31,2026,Maharaja Yadavindra Singh International Cricke...,New Chandigarh,Gujarat Titans,Punjab Kings,None,gujarattitans,punjabkings,2026-03-31__gujarattitans||punjabkings
4,1527678,1527678.json,2026-04-01,2026,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,Lucknow,Lucknow Super Giants,Delhi Capitals,None,lucknowsupergiants,delhicapitals,2026-04-01__delhicapitals||lucknowsupergiants


In [12]:
# Validate the normalized match metadata.

checks = {
    "Rows equal Cricsheet 2026 matches":
        len(normalized_match_metadata) == len(cricsheet_2026),

    "Unique match IDs":
        normalized_match_metadata["match_id"].is_unique,

    "No missing match IDs":
        normalized_match_metadata["match_id"].notna().all(),

    "No missing dates":
        normalized_match_metadata["date"].notna().all(),

    "No missing team 1":
        normalized_match_metadata["team_1"].notna().all(),

    "No missing team 2":
        normalized_match_metadata["team_2"].notna().all(),

    "Unique match keys":
        normalized_match_metadata["match_key"].is_unique,
}

validation_df = pd.DataFrame(
    list(checks.items()),
    columns=["Validation", "Passed"]
)

display(validation_df)

assert validation_df["Passed"].all(), (
    "At least one Day 2 match-metadata validation failed."
)

print("Match metadata validation: PASS")


,Validation,Passed
0,Rows equal Cricsheet 2026 matches,True
1,Unique match IDs,True
2,No missing match IDs,True
3,No missing dates,True
4,No missing team 1,True
5,No missing team 2,True
6,Unique match keys,True


Match metadata validation: PASS


## 8. Day 2 output

At this stage the validated output is:

```text
data/
└── processed/
    └── match_metadata_2026.csv
```

This is a processed artifact. The raw source remains unchanged.

The next production step is to move the validated logic into reusable modules under:

```text
pipeline/
├── extract.py
├── validate.py
├── clean.py
├── transform.py
├── features.py
├── load.py
└── run_pipeline.py
```


# Day 2 stopping point

Before moving to Day 3, verify:

- duplicate investigation output
- 74 vs 70 match reconciliation
- 2026 Cricsheet metadata
- team-name comparison
- player identity inspection
- match metadata validation
- generated `data/processed/match_metadata_2026.csv`

Do not start player-match feature engineering until these validations pass.

### Day 3

Day 3 will build the standardized:

- batting player-match dataset
- bowling player-match dataset
- player identity mapping
- match/date/season/venue/opposition fields
- documented schemas
